In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
# =========================================================
# SETUP
# =========================================================
import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import torch

from src.data.image_processing import HousingImageDataset
from src.training.train_cnn import train_cnn
from src.training.extract_condition_score import extract_condition_scores


# =========================================================
# DEVICE CHECK
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"

print("===================================")
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("Running on CPU")

print("===================================")


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv("../data/processed/dataset_c_cleaned.csv")
IMAGE_PATH = "../data/raw/images"

print(f"Dataset size: {df.shape}")


# =========================================================
# CREATE DATASET
# =========================================================
dataset = HousingImageDataset(df, IMAGE_PATH)

print(f"Total images: {len(dataset)}")


# =========================================================
# TRAIN CNN
# =========================================================
print("\nStarting CNN training...\n")

cnn_model = train_cnn(
    df,
    IMAGE_PATH,
    epochs=5,
    batch_size=32
)
os.makedirs("../outputs/models", exist_ok=True)

torch.save(cnn_model.state_dict(), "../outputs/models/cnn_model.pth")

print("CNN model saved successfully.")


# =========================================================
# EXTRACT CONDITION SCORES
# =========================================================
print("\nExtracting condition scores...\n")

condition_scores = extract_condition_scores(cnn_model, dataset)

print("Condition score shape:", condition_scores.shape)


# =========================================================
# SAVE OUTPUT
# =========================================================
np.save("../data/processed/condition_scores.npy", condition_scores)

print("\nSaved condition scores to:")
print("../data/processed/condition_scores.npy")

Using device: cuda
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA Version: 12.1
Dataset size: (14338, 9)
Total images: 14338

Starting CNN training...



c:\Users\brant\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\brant\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1, Loss: 2571.3418
Epoch 2, Loss: 55.2567
Epoch 3, Loss: 34.0624
Epoch 4, Loss: 21.1113
Epoch 5, Loss: 14.4659
CNN model saved successfully.

Extracting condition scores...



100%|██████████| 449/449 [02:13<00:00,  3.36it/s]

Condition score shape: (14338, 1)

Saved condition scores to:
../data/processed/condition_scores.npy
